<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [1]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [2]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [3]:
# Write your code here
# 1. Display general information about the dataframe (types, counts, memory)
print("--- General Dataframe Summary ---")
df.info()

# 2. Display the data types of each column
print("\n--- Column Data Types ---")
print(df.dtypes)

# 3. Count the number of missing values (NaN) in each column
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

--- General Dataframe Summary ---
<class 'pandas.DataFrame'>
RangeIndex: 65437 entries, 0 to 65436
Columns: 114 entries, ResponseId to JobSat
dtypes: float64(13), int64(1), str(100)
memory usage: 56.9 MB

--- Column Data Types ---
ResponseId               int64
MainBranch                 str
Age                        str
Employment                 str
RemoteWork                 str
                        ...   
JobSatPoints_11        float64
SurveyLength               str
SurveyEase                 str
ConvertedCompYearly    float64
JobSat                 float64
Length: 114, dtype: object

--- Missing Values Count ---
ResponseId                 0
MainBranch                 0
Age                        0
Employment                 0
RemoteWork             10631
                       ...  
JobSatPoints_11        35992
SurveyLength            9255
SurveyEase              9199
ConvertedCompYearly    42002
JobSat                 36311
Length: 114, dtype: int64


<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [5]:
# Write your code 
print("--- Summary Statistics for Numerical Columns ---")
numerical_summary = df.describe()

# Display the summary
print(numerical_summary)

--- Summary Statistics for Numerical Columns ---
         ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [6]:
# Write your code here
# 1. List all unique values in the 'Country' column to spot typos
print("--- Unique Values in 'Country' ---")
unique_countries = df['Country'].unique()
print(unique_countries)

# 2. Get counts of each unique value, sorted alphabetically
# Sorting by index (the country name) makes it easier to find similar-looking typos
print("\n--- Value Counts sorted by Name ---")
country_counts = df['Country'].value_counts().sort_index()
print(country_counts)

# 3. Example: Filter for a specific suspected inconsistent value
# Replace 'SuspectedValue' with an actual value found in step 1 or 2
# print("\n--- Rows with suspected inconsistent value ---")
# print(df[df['Country'] == 'SuspectedValue'])

--- Unique Values in 'Country' ---
<StringArray>
[                            'United States of America',
 'United Kingdom of Great Britain and Northern Ireland',
                                               'Canada',
                                               'Norway',
                                           'Uzbekistan',
                                               'Serbia',
                                               'Poland',
                                          'Philippines',
                                             'Bulgaria',
                                          'Switzerland',
 ...
                                'Saint Kitts and Nevis',
                                               'Monaco',
                   'Micronesia, Federated States of...',
                                                'Haiti',
                                                    nan,
                                                'Nauru',
                                  

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [7]:
## Write your code here
# 1. Define a dictionary for standardization (Mapping)
# Key = Current 'messy' value, Value = New 'clean' value
country_mapping = {
    'United States of America': 'USA',
    'U.S.A.': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK'
}

ed_level_mapping = {
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 'Bachelor',
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 'Master'
}

# 2. Apply the mapping to the dataframe
# We assign the result back to the column instead of using inplace=True
df['Country'] = df['Country'].replace(country_mapping)
df['EdLevel'] = df['EdLevel'].replace(ed_level_mapping)

# 3. Verify the changes (showing top entries)
print("--- Standardized Value Counts for Country ---")
print(df['Country'].value_counts().head())

print("\n--- Standardized Value Counts for EdLevel ---")
print(df['EdLevel'].value_counts().head())

--- Standardized Value Counts for Country ---
Country
USA        11095
Germany     4947
India       4231
UK          3224
Ukraine     2672
Name: count, dtype: int64

--- Standardized Value Counts for EdLevel ---
EdLevel
Bachelor                                                                              24942
Master                                                                                15557
Some college/university study without earning a degree                                 7651
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     5793
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         2970
Name: count, dtype: int64


### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [8]:
## Write your code here
# 1. Perform one-hot encoding on the 'Employment' column
# This creates new binary columns and removes the original 'Employment' column
df_encoded = pd.get_dummies(df, columns=['Employment'])

# 2. Display the names of the newly created columns
print("--- New Columns after One-Hot Encoding ---")
employment_cols = [col for col in df_encoded.columns if col.startswith('Employment')]
print(employment_cols)

# 3. Preview the first few rows of the transformed dataframe
print("\n--- Transformed Dataframe Preview ---")
print(df_encoded.head())

# Save the transformed dataframe to a CSV file
df_encoded.to_csv('encoded_survey_data.csv', index=False)

--- New Columns after One-Hot Encoding ---
['Employment_Employed, full-time', 'Employment_Employed, full-time;Employed, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time;Retired', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work;Employed, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work;Student, part-time', 'Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Retired', 'Employment_Employed, full-time;Independent con

### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [9]:
## Write your code here
# 1. Count missing values for each column
# 2. Sort them in descending order (highest first)
# 3. Display the top 10 columns
print("--- Columns with the Most Missing Values ---")
missing_values_sorted = df.isnull().sum().sort_values(ascending=False)

print(missing_values_sorted.head(10))

--- Columns with the Most Missing Values ---
AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64


<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [10]:
## Write your code here
# 1. Check the number of missing values before fixing
print("--- Missing Values in 'ConvertedCompYearly' (Before) ---")
print(df['ConvertedCompYearly'].isnull().sum())

# 2. Calculate the Median
# We use median because it is robust against outliers (like extreme high salaries)
median_value = df['ConvertedCompYearly'].median()
print(f"\nCalculated Median: {median_value}")

# 3. Impute (Fill) the missing values
# We assign the result back to the column instead of using inplace=True
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(median_value)

# 4. Verify the fix
print("\n--- Missing Values in 'ConvertedCompYearly' (After) ---")
print(df['ConvertedCompYearly'].isnull().sum())

--- Missing Values in 'ConvertedCompYearly' (Before) ---
42002

Calculated Median: 65000.0

--- Missing Values in 'ConvertedCompYearly' (After) ---
0


<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [11]:
## Write your code here
# 1. Identify the most frequent value (Mode)
# .mode() returns a Series, so we take the first element [0]
mode_value = df['RemoteWork'].mode()[0]
print(f"Most frequent value (Mode): {mode_value}")

# 2. Check missing values before imputation
print("\nMissing values in 'RemoteWork' (Before):")
print(df['RemoteWork'].isnull().sum())

# 3. Fill missing values with the mode
# We assign back to the column instead of using inplace=True
df['RemoteWork'] = df['RemoteWork'].fillna(mode_value)

# 4. Verify the results
print("\nMissing values in 'RemoteWork' (After):")
print(df['RemoteWork'].isnull().sum())

Most frequent value (Mode): Hybrid (some remote, some in-person)

Missing values in 'RemoteWork' (Before):
10631

Missing values in 'RemoteWork' (After):
0


### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [12]:
## Write your code here
# 1. Calculate the minimum and maximum values of the column
min_val = df['ConvertedCompYearly'].min()
max_val = df['ConvertedCompYearly'].max()

# 2. Apply the Min-Max formula: (x - min) / (max - min)
# We create a new column to store the normalized values
df['ConvertedCompYearly_normalized'] = (df['ConvertedCompYearly'] - min_val) / (max_val - min_val)

# 3. Verify the transformation
print("--- Normalized Column Statistics ---")
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_normalized']].describe())

# Preview the first few rows
print("\n--- Dataframe Preview (Normalized) ---")
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_normalized']].head())

--- Normalized Column Statistics ---
       ConvertedCompYearly  ConvertedCompYearly_normalized
count         6.543700e+04                    65437.000000
mean          7.257636e+04                        0.004464
std           1.122207e+05                        0.006903
min           1.000000e+00                        0.000000
25%           6.500000e+04                        0.003998
50%           6.500000e+04                        0.003998
75%           6.500000e+04                        0.003998
max           1.625660e+07                        1.000000

--- Dataframe Preview (Normalized) ---
   ConvertedCompYearly  ConvertedCompYearly_normalized
0              65000.0                        0.003998
1              65000.0                        0.003998
2              65000.0                        0.003998
3              65000.0                        0.003998
4              65000.0                        0.003998


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [13]:
## Write your code here
import numpy as np

# 1. Apply Log Transformation using NumPy
# We use np.log() for the natural logarithm
# Following the rule, we assign the result to a new column
df['ConvertedCompYearly_log'] = np.log(df['ConvertedCompYearly'])

# 2. Compare the skewness of the original vs transformed data
# A skewness value closer to 0 indicates a more symmetrical distribution
print("--- Skewness Comparison ---")
print(f"Original Skewness: {df['ConvertedCompYearly'].skew():.2f}")
print(f"Log-transformed Skewness: {df['ConvertedCompYearly_log'].skew():.2f}")

# 3. Preview the transformation
print("\n--- Dataframe Preview (Log-transformed) ---")
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_log']].head())

--- Skewness Comparison ---
Original Skewness: 87.71
Log-transformed Skewness: -4.38

--- Dataframe Preview (Log-transformed) ---
   ConvertedCompYearly  ConvertedCompYearly_log
0              65000.0                11.082143
1              65000.0                11.082143
2              65000.0                11.082143
3              65000.0                11.082143
4              65000.0                11.082143


### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [14]:
## Write your code here
# 1. Handle non-numeric entries and convert to numeric
# 'Less than 1 year' becomes 0, 'More than 50 years' becomes 51
df['YearsCodePro'] = df['YearsCodePro'].replace({'Less than 1 year': '0', 'More than 50 years': '51'})
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')

# 2. Define a function for categorization logic
def get_experience_level(years):
    if pd.isna(years):
        return 'Unknown'
    if years < 5:
        return 'Junior'
    elif 5 <= years <= 15:
        return 'Intermediate'
    else:
        return 'Senior'

# 3. Apply the function to create the new column
df['ExperienceLevel'] = df['YearsCodePro'].apply(get_experience_level)

# 4. Verify the distribution of the new categories
print("--- Distribution of Experience Levels ---")
print(df['ExperienceLevel'].value_counts())

--- Distribution of Experience Levels ---
ExperienceLevel
Intermediate    23112
Junior          16971
Unknown         13827
Senior          11527
Name: count, dtype: int64


### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
